**MetroPT-3 Window Labeling and Cleaning Pipeline**

Turns the windowed MetroPT-3 air compressor sensor data into a clean, training-ready dataset: fixed-size windows of 30 rows, correct failure labels, and no frozen-sensor periods.

| Step | Purpose | Input | Output |
|------|---------|-------|--------|
| 1 | Build the `ground_truth` label from the failure events table | `data_modwindowed.csv`, `failure_events_clean (2).csv` | `data_cleaned.csv` |
| 2 | Diagnose why `window_id` cannot be compared numerically | `data_modwindowed.csv` | report only |
| 3 | Rebuild `window_id` as a numeric, increasing id | `data_modwindowed.csv` | `data_windowed_fixed.csv` |
| 4 | Remove frozen-sensor periods | `data_windowed_fixed.csv` | `data_windowed_no_freeze.csv` |
| 5 | Keep only complete windows (30 rows) | `data_windowed_no_freeze.csv` | `data_windowed_final.csv` |
| 6 | Validate the final file | `data_windowed_final.csv` | report only |

Design note: removing frozen periods before windowing would avoid creating incomplete windows that Step 5 has to drop. The current order is kept to match the existing workflow.

**Project files overview**

The descriptions below are inferred from the file names. Adjust any row whose purpose differs.

| File | Type | Purpose |
|------|------|---------|
| `data_raw`, `MetroPT3(AirCompressor)` | CSV | Original raw sensor data of the air compressor |
| `raw_data_loading` | Python | Loads the raw data for the first time |
| `Sentinel_Data_Cleaning_Pipeline` | Python | Main cleaning pipeline |
| `data_cleaned`, `data_cleaned final file` | CSV | Cleaned data including the `ground_truth` label |
| `data_imputed`, `missing_values_imputation`, `imputation_policy`, `missing_report` | Excel / Python | Missing value handling and its reports |
| `timestamp_gaps`, `timestamp_log`, `timestamp_segmentsummary`, `time_stamp`, `ts_fixed`, `data_ts_fixed` | CSV / Python | Timestamp gap detection, fixing and segment summaries |
| `Windowing`, `Windowing (2)`, `windowing_config`, `windows_index`, `data_windowed` | Python / JSON / CSV| Window creation, its configuration and index |
| `data_labeling`, `data_labeling (3)`, `data_labeled`, `failure_events_clean`, `official_failure_check` | Python / CSV | Label creation and verification against official failure events |
| `frozen_periods`, `eda_report_frozen_logger` | CSV / Python | Detection and logging of frozen-sensor periods |
| `sensor_baseline`, `sensor_baseline_by_state`, `sensor_summary`, `sensor_thresholds_by_state`, `baseline_Normal_Data`, `baseline_integrity_summary`, `baseline_quality_report` | Excel / Python / JSON | Normal-state baseline statistics and sensor thresholds |
| `outlier_summary_by_state` | Excel | Outlier summary per operating state |
| `incomplete_parts_summary` | Excel | Summary of incomplete data parts |
| `data_minute_features` | Excel | Features aggregated at minute level |
| `random_forest_model` | Python | Random Forest prediction model |
| `validation_report`, `valid`, `Phase1_Final_Validation_adapted2` | Python / SCV | Phase 1 quality validation |
| `sentinel_output` | Folder | Final project outputs |

**Step 0 - Configuration and shared helpers**

Defines all paths, constants and sensor lists in one place, plus two small helpers: `header` prints a section banner and `load` reads a CSV with a parsed `timestamp` column. Change `BASE` if the project folder moves.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

BASE = Path(r"C:\DEPI_project-main\DEPI_project-main\src\data_processing")

F_SOURCE = BASE / "data_modwindowed.csv"
F_EVENTS = BASE / "failure_events_clean (2).csv"
F_LABELED = BASE / "data_cleaned.csv"
F_FIXED = BASE / "data_windowed_fixed.csv"
F_NOFREEZE = BASE / "data_windowed_no_freeze.csv"
F_FINAL = BASE / "data_windowed_final.csv"

WINDOW_SIZE = 30
MIN_FROZEN = pd.Timedelta(minutes=5)

ANALOG = ["TP2", "TP3", "H1", "DV_pressure", "Reservoirs",
          "Oil_temperature", "Motor_current"]
DIGITAL = ["COMP", "DV_eletric", "Towers", "MPG", "LPS",
           "Pressure_switch", "Oil_level", "Caudal_impulses"]
SENSORS = ANALOG + DIGITAL
OUT_COLS = ["timestamp"] + SENSORS + ["segment", "ground_truth", "window_id"]


def header(title: str) -> None:
    """Print a section banner."""
    print("=" * 60)
    print(title)
    print("=" * 60)


def load(path: Path) -> pd.DataFrame:
    """Read a CSV file and parse its timestamp column."""
    return pd.read_csv(path, parse_dates=["timestamp"])

**Step 1 - Build the `ground_truth` label**

What it does:
- Reads the sensor data and the failure events table (`start_time` / `end_time`).
- Marks every row that falls inside a failure interval with `ground_truth = 1`; all other rows stay `0`.
- Prints how many rows match each event, so an event outside the data range is spotted immediately.
- Saves `data_cleaned.csv` only when at least one positive row exists and all expected columns are present.

In [ ]:
def build_ground_truth(df: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of df with ground_truth = 1 inside any failure interval."""
    df = df.copy()
    df["ground_truth"] = 0
    matched = 0
    for _, ev in events.iterrows():
        mask = df["timestamp"].between(ev["start_time"], ev["end_time"])
        n = int(mask.sum())
        matched += n > 0
        print(f"   {'OK  ' if n else 'MISS'} event {ev['event_id']}: {n:,} rows")
        df.loc[mask, "ground_truth"] = 1
    print(f"\n   matched events: {matched}/{len(events)}")
    return df


header("Step 1: ground_truth labeling")
df = load(F_SOURCE)
events = pd.read_csv(F_EVENTS, parse_dates=["start_time", "end_time"])

print(f"Columns : {list(df.columns)}")
print(f"Rows    : {len(df):,}")
print(f"Range   : {df['timestamp'].min()}  ->  {df['timestamp'].max()}\n")
print(events[["event_id", "start_time", "end_time"]].to_string(index=False), "\n")

df = build_ground_truth(df, events)
pos = int(df["ground_truth"].sum())
print(f"   positives = {pos:,} / {len(df):,} ({pos / len(df):.4%})")

if pos == 0:
    print("\nWARNING: no row has ground_truth = 1. Check that the event dates fall inside the data range.")
else:
    missing = [c for c in OUT_COLS if c not in df.columns]
    if missing:
        print(f"\nWARNING: missing columns: {missing}")
    else:
        df[OUT_COLS].to_csv(F_LABELED, index=False)
        print(f"\nSaved: {F_LABELED.name}  ({len(df):,} rows x {len(OUT_COLS)} columns)")

**Step 2 - Diagnose the `window_id` column**

What it does:
- Checks that `timestamp` is sorted in increasing order.
- Inspects the type of `window_id`. In this dataset it is a string such as `segment_0_window_1`, not a number. That is why `.diff()` raised `TypeError`, and why `pd.to_numeric(..., errors="coerce")` only produces NaN values and makes the "backward jumps" count meaningless.
- Checks that each window belongs to exactly one segment.

Conclusion: the string ids cannot be ordered numerically, so `window_id` is rebuilt in Step 3.

In [ ]:
header("Step 2: window_id diagnosis")
df = load(F_SOURCE)

print("timestamp increasing :", df["timestamp"].is_monotonic_increasing)
print("window_id dtype      :", df["window_id"].dtype)
print("window_id nulls      :", int(df["window_id"].isna().sum()))
print("sample values        :", df["window_id"].astype(str).unique()[:5].tolist())

numeric = pd.to_numeric(df["window_id"], errors="coerce")
print(f"numeric-valid values : {numeric.notna().sum():,} / {len(df):,}")

segments_per_window = df.groupby("window_id")["segment"].nunique()
bad = segments_per_window[segments_per_window > 1]
if bad.empty:
    print("\nwindows spanning more than one segment: none")
else:
    print(f"\nwindows spanning more than one segment: {len(bad)}, e.g. {list(bad.index[:5])}")

**Step 3 - Rebuild `window_id` (numeric, increasing, inside a segment)**

What it does:
1. Sorts rows by `segment`, then `timestamp`.
2. Drops the old string `window_id`.
3. Splits every segment into windows of 30 rows (`row position inside the segment // 30`).
4. Converts the local window number into a global sequential id using a cumulative offset per segment.

The last window of a segment can be shorter than 30 rows; Step 5 removes it.

In [ ]:
def rebuild_window_ids(df: pd.DataFrame, size: int = WINDOW_SIZE) -> pd.DataFrame:
    """Return df with a new numeric, globally increasing window_id."""
    df = (df.drop(columns=["window_id"], errors="ignore")
            .sort_values(["segment", "timestamp"], kind="stable")
            .reset_index(drop=True))
    local = df.groupby("segment").cumcount() // size
    windows_per_segment = local.groupby(df["segment"]).max() + 1
    offset = windows_per_segment.cumsum() - windows_per_segment
    df["window_id"] = (local + df["segment"].map(offset)).astype(int)
    return df


header("Step 3: rebuild window_id")
df = rebuild_window_ids(load(F_SOURCE))

assert df["window_id"].is_monotonic_increasing, "window_id must be increasing"
print(f"segments : {df['segment'].nunique():,}")
print(f"windows  : {df['window_id'].nunique():,}")
print(f"min/max  : {df['window_id'].min()} / {df['window_id'].max()}")

df.to_csv(F_FIXED, index=False)
print(f"\nSaved: {F_FIXED.name} ({len(df):,} rows)")

**Step 4 - Remove frozen-sensor periods**

What it does:
- A row is "frozen" when all seven analog sensors are exactly equal to the previous row.
- Consecutive frozen rows are grouped into periods, and only periods lasting 5 minutes or more are kept.
- Every row of those periods, including the row just before the freeze starts, is removed.
- The detected periods are displayed as a table, then the cleaned data is saved.

In [ ]:
def find_frozen_periods(df: pd.DataFrame, cols=ANALOG, min_duration=MIN_FROZEN) -> pd.DataFrame:
    """Return frozen periods (start_pos, end_pos, start_time, end_time, duration) lasting at least min_duration."""
    same = df[cols].eq(df[cols].shift(1)).all(axis=1)
    same.iloc[0] = False
    run_id = same.ne(same.shift(fill_value=False)).cumsum()

    rows = []
    for _, idx in df.index[same].to_series().groupby(run_id[same]):
        start_pos = max(idx.iloc[0] - 1, 0)
        end_pos = idx.iloc[-1]
        start = df.at[start_pos, "timestamp"]
        end = df.at[end_pos, "timestamp"]
        if end - start >= min_duration:
            rows.append((start_pos, end_pos, start, end, end - start))
    return pd.DataFrame(rows, columns=["start_pos", "end_pos", "start_time",
                                       "end_time", "duration"])


header("Step 4: remove frozen periods")
df = load(F_FIXED).sort_values("timestamp", kind="stable").reset_index(drop=True)
print(f"Before: {len(df):,} rows")

frozen = find_frozen_periods(df)
print(f"Frozen periods: {len(frozen)}")
display(frozen[["start_time", "end_time", "duration"]])

drop = np.zeros(len(df), dtype=bool)
for start_pos, end_pos in zip(frozen["start_pos"], frozen["end_pos"]):
    drop[start_pos:end_pos + 1] = True

print(f"Rows to remove: {drop.sum():,} ({drop.mean():.2%})")
df = df[~drop].reset_index(drop=True)
print(f"After : {len(df):,} rows")

df.to_csv(F_NOFREEZE, index=False)
print(f"\nSaved: {F_NOFREEZE.name}")

**Step 5 - Keep complete windows only**

What it does:
- After the frozen periods and the short tail window of each segment are removed, some windows have fewer than 30 rows.
- Counts the rows of every window and keeps only windows with exactly 30 rows, because the model needs fixed-size inputs.
- Saves the result as `data_windowed_final.csv`.

In [ ]:
header("Step 5: keep complete windows only")
df = load(F_NOFREEZE)

sizes = df.groupby("window_id").size()
full_windows = sizes[sizes == WINDOW_SIZE].index
df_final = df[df["window_id"].isin(full_windows)].reset_index(drop=True)

print(f"Before : {len(df):,} rows, {len(sizes):,} windows")
print(f"After  : {len(df_final):,} rows, {len(full_windows):,} windows (all {WINDOW_SIZE} rows)")
print(f"Removed: {len(df) - len(df_final):,} rows (incomplete windows)")

df_final.to_csv(F_FINAL, index=False)
print(f"\nSaved: {F_FINAL.name}")

**Step 6 - Validate the final file**

Confirms the conditions training relies on:
- every window has exactly 30 rows;
- every window belongs to a single segment;
- `window_id` is increasing;
- there are no missing values in the sensor columns;
- the share of positive `ground_truth` rows and windows.

If `ground_truth` is missing from the final file, run Step 1 on the source file before Step 3 so the label is carried through.

In [ ]:
header("Step 6: final validation")
df = load(F_FINAL)

checks = {
    "all windows have 30 rows": bool((df.groupby("window_id").size() == WINDOW_SIZE).all()),
    "each window is in one segment": bool((df.groupby("window_id")["segment"].nunique() == 1).all()),
    "window_id is increasing": bool(df["window_id"].is_monotonic_increasing),
    "no nulls in sensor columns": bool(df[SENSORS].notna().all().all()),
}
for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

if "ground_truth" in df.columns:
    window_label = df.groupby("window_id")["ground_truth"].max()
    print(f"\npositive rows   : {df['ground_truth'].mean():.4%}")
    print(f"positive windows: {window_label.mean():.4%} ({int(window_label.sum()):,} / {len(window_label):,})")
else:
    print("\nWARNING: ground_truth column is not present in the final file.")